In [ ]:
from networkx.convert import to_networkx_graph
from torch_geometric.utils import to_networkx

import AttackerGNN.MainAttack
import config
%cd {config.PROJECT_DIR}

import sys
print(sys.executable)
print(sys.version)

In [ ]:
DATASET = 'cora_ml'

In [ ]:
import os
files = ["cache/demo.json", "cache/demo/demo_1.pt", "cache/evasion_global_adj.json", "cache/evasion_global_attr.json", "cache/evasion_global_adj/evasion_global_adj_1.pt", "cache/evasion_global_attr/evasion_global_attr_1.pt"]

for file_path in files:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")
    else:
        print(f"{file_path} does not exist.")

In [ ]:
SEED = 1

In [ ]:
#7 run
#%cd robustness_of_gnns_at_scale
from matplotlib import pyplot as plt

from experiments import (
    experiment_train,
    experiment_local_attack_direct,
    experiment_global_attack_direct
)

%matplotlib inline

%load_ext autoreload
%autoreload 2

## 2. Train Victim Model

In [ ]:
train_statistics = experiment_train.run(
    data_dir = './data',
    dataset = DATASET,
    model_params = dict(
        label="Vanilla GCN",
        model="GCN",
        do_cache_adj_prep=True,
        n_filters=64,
        dropout=0.5,
        svd_params=None,
        jaccard_params=None,
        gdc_params={"alpha": 0.15, "k": 64}),
    train_params = dict(
        lr=1e-2,
        weight_decay=1e-3,
        patience=300,
        max_epochs=3000),
    binary_attr = False,
    make_undirected = True,
    seed=SEED,
    artifact_dir = 'cache',
    model_storage_type = 'demo',
    ppr_cache_params = dict(),
    #device = 0,
    #data_device = 0,
    device = "cpu",
    data_device = "cpu",
    display_steps = 100,
    debug_level = "info"
)

# plot train and val loss curves
fig, ax = plt.subplots()

trained_model = train_statistics['model']
graph = train_statistics["graph"]

from rgnn_at_scale.attacks.prbcd_miss_experiment import graph_to_attack_inputs

adj, attr, labels = graph_to_attack_inputs(
    graph,
    data_device="cpu",
)

print(adj.sparse_sizes())
print(attr.shape, attr.dtype)
print(labels.shape, labels.dtype)

color = plt.rcParams['axes.prop_cycle'].by_key()['color'][0]
ax.set_xlabel('Epoch $t$')
ax.set_ylabel("Loss")
ax.plot(train_statistics['trace_train'], color=color, label='Train')

color = plt.rcParams['axes.prop_cycle'].by_key()['color'][1]
ax.plot(train_statistics['trace_val'], color=color, label='Val')
ax.legend()
plt.show()

clean_acc = train_statistics["accuracy"]
print(f'Accuracy of the model: {100*clean_acc:.2f}%')

In [ ]:
# certificate integration
import torch
import numpy as np
import copy
from sparse_smoothing.models import GCN
from sparse_smoothing.utils import load_and_standardize

loaded_from_robustness = torch.load(f'cache/demo/demo_1.pt', map_location='cpu', weights_only=False)

trained_state_dict = copy.deepcopy(loaded_from_robustness)

for key in list(trained_state_dict.keys()):
    if 'layers.0.' in key:
        new_key = key.replace('layers.0.gcn_0', 'conv1')
        trained_state_dict[new_key] = trained_state_dict.pop(key)  # Transpose if necessary

for key in list(trained_state_dict.keys()):
    if 'layers.1.' in key:
        new_key = key.replace('layers.1.gcn_1', 'conv2')
        trained_state_dict[new_key] = trained_state_dict.pop(key)  # Transpose if necessary

In [ ]:
graph = load_and_standardize(f"data/{DATASET}.npz")
n, d = graph.attr_matrix.shape
nc = graph.labels.max() + 1
pf_plus_att = 0.01
pf_minus_att = 0.6

In [ ]:
model = GCN(n_features=d, n_classes=nc, n_hidden=64)#.cuda()
print(model)
# Load the modified state dictionary
model.load_state_dict(trained_state_dict)

In [ ]:
from torch_geometric.utils import to_networkx
import networkx as nx

adj_numpy = graph.adj_matrix.toarray()
y_numpy = graph.labels

# Convert numpy adjacency matrix to networkx graph
G = nx.from_numpy_array(adj_numpy)

#G = to_networkx(graph.adj_matrix.toarray(), to_undirected=True)
plt.figure(figsize=(10, 10))
plt.axis('off')
nx.draw(G, with_labels=True, node_color='skyblue', node_size=500, font_size=15)
plt.show()
print(adj_numpy.shape)

In [2]:
print(adj_numpy.shape)
print(y_numpy.shape)

edge_idx = torch.LongTensor(np.stack(graph.adj_matrix.nonzero()))#.cuda()
attr_idx = torch.LongTensor(np.stack(graph.attr_matrix.nonzero()))#.cuda()

NameError: name 'adj_numpy' is not defined

In [ ]:
# =========================
# Sweep config: PR-BCD / ATTACK PARAMS
# =========================

results_global_prbcd_cert = []

attack_sweep = {
    # experiment-level attack settings
    "semi": [True],
    "epsilon": [0.05],
    "use_cert": [
        "accuracy_drop_selector",
    ],

    # PR-BCD-internal params
    "block_size": [100_000],
    "epochs": [100],
    "fine_tune_epochs": [50],
    "keep_heuristic": ["WeightOnly"],
    "search_space_size": [100_000],
    "do_synchronize": [True],
    "loss_type": ["tanhMargin"],
}

# =========================
# Sweep config: SELECTOR / PRESELECTION PARAMS
# =========================

selector_sweep = {
    # ADS-specific selector/preselection mode
    # These should ONLY be swept for use_cert in ads_certs.
    "accuracy_drop_selector_mode": [
        "one_sample",
    ],  # "k_hop", "one_sample", "k_action", "node_matching"

    #Todo: Victim GCN may only affect the 2-hop neighborhood. simultaneous flipping 2x2-hop neighborhood.

    # ADS-specific candidate labeling
    # These should ONLY be assigned for use_cert in ads_certs.
    "n_candidates_k_sample": [2000],
    "n_candidates_one_sample": [5000],
    "drop_mode": ["endpoint"],
    "acc_drop_threshold_k_samples": [1e-3],
    "loss_drop_threshold_k_samples": [1e-3],
    "k_samples_batch": [10],
    "training_data_node_cap": [15],

    # LP / selector block construction
    # These can still be passed for non-ADS configs if you want them available.
    "tau": [0.8],
    "score_batch_size": [1000],
    "max_sampling_tries": [2_000_000],
    "exclude_tried": [True],

    "lp_hit_rate_detour": [False],
    "lp_hit_rate_top_k": [200],
    "lp_hit_rate_out_dir": ["extendedPlotting/lpEndpointHitRate"],



    # optional / future params
    # "subgraph_fraction": [0.33],
    # "tau_decay": [None],
    # "aux_loss_weight": [None],
}

# =========================
# Sweep definitions
# =========================

from itertools import product
from copy import deepcopy

ads_certs = {
    "accuracy_drop_selector",
    "accuracy_drop_selector_with_resampling",
    "selector_direct"
}

# Params that should only exist for ADS configs
ADS_ONLY_SELECTOR_COLUMNS = {
    "accuracy_drop_selector_mode",
    "n_candidates_k_sample",
    "n_candidates_one_sample",
    "drop_mode",
    "acc_drop_threshold_k_samples",
    "loss_drop_threshold_k_samples",
    "k_samples_batch",
    "training_data_node_cap",
}

# Params that may still be useful/loggable for non-ADS configs
BASE_SELECTOR_COLUMNS = [
    col for col in selector_sweep.keys()
    if col not in ADS_ONLY_SELECTOR_COLUMNS
]

ATTACK_COLUMNS = list(attack_sweep.keys())
SELECTOR_COLUMNS = list(selector_sweep.keys())

ALL_SWEEP_COLUMNS = (
    [f"attack__{col}" for col in ATTACK_COLUMNS]
    + [f"selector__{col}" for col in SELECTOR_COLUMNS]
)


def expand_grid(grid):
    keys = list(grid.keys())
    value_lists = [grid[k] for k in keys]

    for values in product(*value_lists):
        yield dict(zip(keys, values))


def filter_grid(grid, keep_columns):
    """
    Keeps only selected columns from a sweep grid.
    """
    return {
        key: values
        for key, values in grid.items()
        if key in keep_columns
    }


def build_sweep_configs():
    configs = []

    ads_selector_sweep = selector_sweep
    base_selector_sweep = filter_grid(
        selector_sweep,
        BASE_SELECTOR_COLUMNS,
    )

    for attack_cfg in expand_grid(attack_sweep):
        use_cert = attack_cfg["use_cert"]

        if use_cert in ads_certs:
            # ADS configs:
            # Sweep over accuracy_drop_selector_mode, drop_mode, candidate labeling,
            # and LP/block-construction params.
            for selector_cfg in expand_grid(ads_selector_sweep):
                configs.append({
                    "attack_params": deepcopy(attack_cfg),
                    "selector_params": deepcopy(selector_cfg),
                })

        else:
            # Non-ADS configs:
            # Do NOT sweep or assign accuracy_drop_selector_mode or drop_mode.
            # Still pass base selector params like tau, score_batch_size,
            # max_sampling_tries, exclude_tried.
            for selector_cfg in expand_grid(base_selector_sweep):
                configs.append({
                    "attack_params": deepcopy(attack_cfg),
                    "selector_params": deepcopy(selector_cfg),
                })

    return configs


def _cfg_to_row(cfg):
    row = {}

    attack_params = cfg.get("attack_params", {})
    selector_params = cfg.get("selector_params", {})

    for col in ATTACK_COLUMNS:
        row[f"attack__{col}"] = attack_params.get(col, "")

    for col in SELECTOR_COLUMNS:
        row[f"selector__{col}"] = selector_params.get(col, "")

    return row


sweep_configs = build_sweep_configs()
number_loops = len(sweep_configs)

print("number_loops:", number_loops)

# Optional sanity check
for i, cfg in enumerate(sweep_configs, start=1):
    attack_params = cfg["attack_params"]
    selector_params = cfg["selector_params"]

    use_cert = attack_params["use_cert"]
    eps = attack_params["epsilon"]
    ads_mode = selector_params.get("accuracy_drop_selector_mode", "")
    drop_mode = selector_params.get("drop_mode", "")
    node_cap = selector_params.get("training_data_node_cap", "")

    print(
        i,
        "use_cert:", use_cert,
        "epsilon:", eps,
        "ads_mode:", ads_mode,
        "drop_mode:", drop_mode,
        "node_cap:", node_cap,
        "selector_params:", selector_params,
    )

In [ ]:
from rgnn_at_scale.attacks.prbcd import PRBCD

from rgnn_at_scale.attacks.prbcd_miss_experiment import (
    PRBCDLoopConfig,
    build_attack_factory,
    compare_prbcd_runs,
    concise_report,
)


make_attack = build_attack_factory(
    PRBCD,

    # Your trained victim and clean graph:
    model=trained_model,
    adj=adj,
    attr=attr,
    labels=labels,
    idx_attack=idx_attack,

    # Devices:
    device=device,
    data_device=device,

    # Base attack settings:
    make_undirected=True,
    binary_attr=False,
    loss_type="tanhMargin",

    # Initial constructor values. The experiment configuration
    # overwrites these for each run.
    epochs=400,
    fine_tune_epochs=100,
    block_size=50_000,
    lr_factor=100,
    with_early_stopping=True,
)


reference_attack = make_attack()
candidate_attack = make_attack()